In [1]:
import polars as pl
import numpy as np
import pandas as pd
import gc
import os
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.feature_selection import mutual_info_classif  # альтернатива, можно LightGBM

print(f'LightGBM: {lgb.__version__}')

LightGBM: 4.6.0


In [2]:
# ── Конфиг ───────────────────────────────────────────────────────────────
DATA_DIR = 'data/'           # поправь под свой путь
OUT_DIR  = 'data/'
N_FOLDS  = 4
SEED     = 42
TOP_K_FEATURES = 400         # количество отбираемых признаков
USE_GPU = False              # включить GPU (если True, нужна поддержка LightGBM GPU)
# ─────────────────────────────────────────────────────────────────────────

## 1. Загрузка данных

In [3]:
# ----------------------------------------------------------------------
# 1. Объединение main + extra в wide (только один раз)
# ----------------------------------------------------------------------
WIDE_TRAIN = f'{DATA_DIR}train_wide_features.parquet'
WIDE_TEST  = f'{DATA_DIR}test_wide_features.parquet'

if not os.path.exists(WIDE_TRAIN):
    print("Создаём wide-таблицы из main + extra...")
    lf_main_train = pl.scan_parquet(f'{DATA_DIR}train_main_features.parquet').drop('customer_id')
    lf_extra_train = pl.scan_parquet(f'{DATA_DIR}train_extra_features.parquet').drop('customer_id')
    lf_main_test = pl.scan_parquet(f'{DATA_DIR}test_main_features.parquet').drop('customer_id')
    lf_extra_test = pl.scan_parquet(f'{DATA_DIR}test_extra_features.parquet').drop('customer_id')

    train_wide = pl.concat([lf_main_train, lf_extra_train], how='horizontal') \
                   .with_columns(pl.all().cast(pl.Float16)).collect()
    test_wide = pl.concat([lf_main_test, lf_extra_test], how='horizontal') \
                  .with_columns(pl.all().cast(pl.Float16)).collect()

    # удалим дубликаты столбцов (если есть)
    def dedup(df):
        seen = set()
        cols = []
        for c in df.columns:
            if c not in seen:
                seen.add(c)
                cols.append(c)
        return df.select(cols)
    train_wide = dedup(train_wide)
    test_wide = dedup(test_wide)

    train_wide.write_parquet(WIDE_TRAIN)
    test_wide.write_parquet(WIDE_TEST)
    print("Wide-файлы сохранены.")
else:
    print("Загружаем готовые wide-файлы...")
    train_wide = pl.read_parquet(WIDE_TRAIN)
    test_wide = pl.read_parquet(WIDE_TEST)


Загружаем готовые wide-файлы...


## 2. Feature Engineering

In [4]:
def feature_engineering(df: pl.DataFrame) -> pl.DataFrame:
    """
    Максимально быстрый feature engineering.
    Пропускаем std, используем только простые агрегаты.
    """
    num_cols = [c for c in df.columns if c.startswith('num_feature')]
    
    # Простые агрегаты без сложных вычислений
    return df.with_columns([
        pl.sum_horizontal(
            *(pl.col(c).is_not_null().cast(pl.Int32) for c in num_cols)
        ).alias('num_not_null_count'),
        pl.mean_horizontal(*[pl.col(c) for c in num_cols]).alias('num_row_mean'),
        pl.max_horizontal(*[pl.col(c) for c in num_cols]).alias('num_row_max'),
        pl.min_horizontal(*[pl.col(c) for c in num_cols]).alias('num_row_min'),
    ])

print("Feature engineering...")
train_wide = feature_engineering(train_wide)
test_wide  = feature_engineering(test_wide)


Feature engineering...


## 3. Отбор топ-K признаков для каждого таргета (кэшируется)

In [ ]:
SELECTED_DIR = f'{DATA_DIR}selected_features'
os.makedirs(SELECTED_DIR, exist_ok=True)

LOAD_BATCH_SIZE = 100_000      # 50k -> 100k (больше данных за раз)
SUBSAMPLE_SIZE = 200_000       # 120k -> 200k (лучше отбор)
TOP_K_FEATURES = 400   

# Загружаем целевые переменные
target = pl.read_parquet(f'{DATA_DIR}train_target.parquet')
target_cols = [c for c in target.columns if c.startswith('target_')]

# Загружаем customer_id отдельно (из исходных файлов, а не из wide)
train_customer_ids = pl.read_parquet(f'{DATA_DIR}train_main_features.parquet').select('customer_id')
test_customer_ids = pl.read_parquet(f'{DATA_DIR}test_main_features.parquet').select('customer_id')

# Определяем категориальные признаки
cat_feature_names = [c for c in train_wide.columns if c.startswith('cat_feature')]

# Приводим категориальные к int32
train_wide = train_wide.with_columns(pl.col(cat_feature_names).cast(pl.Int32))
test_wide = test_wide.with_columns(pl.col(cat_feature_names).cast(pl.Int32))

def select_features_for_target(target_name, X, y, cat_features, k=TOP_K_FEATURES, subsample_size=SUBSAMPLE_SIZE):
    """Отбирает k наиболее важных признаков с помощью LightGBM с контролем размера подвыборки"""
    path = f'{SELECTED_DIR}/{target_name}.npy'
    if os.path.exists(path):
        return np.load(path).tolist()
    
    print(f'  Отбор признаков для {target_name}...')
    
    # Создаем сбалансированную подвыборку
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    
    # Определяем размер подвыборки в зависимости от количества положительных
    if n_pos < subsample_size / 2:
        # Берем все положительные и добираем отрицательными
        n_neg = min(subsample_size - n_pos, len(neg_idx))
        if n_neg > 0:
            idx = np.concatenate([pos_idx, np.random.choice(neg_idx, n_neg, replace=False)])
        else:
            idx = pos_idx
    else:
        # Берем сбалансированную подвыборку
        n_pos_sample = min(subsample_size // 2, n_pos)
        n_neg_sample = min(subsample_size // 2, len(neg_idx))
        pos_sample = np.random.choice(pos_idx, n_pos_sample, replace=False)
        neg_sample = np.random.choice(neg_idx, n_neg_sample, replace=False)
        idx = np.concatenate([pos_sample, neg_sample])
    
    idx = np.sort(idx)
    
    # Загружаем подвыборку
    X_sub = X.iloc[idx]
    y_sub = y[idx]
    
    print(f'    Подвыборка: {len(X_sub)} строк (положительных: {y_sub.sum()})')
    
    # Быстрое обучение LightGBM для важности
    model = lgb.LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        num_leaves=31,
        min_child_samples=10,
        random_state=SEED, 
        n_jobs=-1, 
        verbose=-1
    )
    
    # Категориальные признаки, которые есть в подвыборке
    cat_features_subset = [c for c in cat_features if c in X_sub.columns]
    
    model.fit(X_sub, y_sub, categorical_feature=cat_features_subset)
    
    # Получаем важность признаков
    importance = model.feature_importances_
    top_idx = np.argsort(importance)[::-1][:k]
    selected = X_sub.columns[top_idx].tolist()
    
    # Сохраняем
    np.save(path, np.array(selected))
    print(f'    Отобрано {len(selected)} признаков')
    return selected

# Преобразуем wide в pandas для обучения
print("Конвертация в pandas...")
X_train = train_wide.to_pandas()  # customer_id уже нет
X_test = test_wide.to_pandas()
y_train = target.select(target_cols).to_pandas()
test_customer_ids_list = test_customer_ids['customer_id'].to_list()

print(f"Размер данных: X_train={X_train.shape}, X_test={X_test.shape}")

# Кэшируем отобранные признаки для каждого таргета
selected_features_dict = {}
print("\nОтбор признаков для каждого таргета:")
for i, t in enumerate(target_cols):
    print(f"\n[{i+1}/{len(target_cols)}] {t}...")
    y_t = y_train[t].values
    
    # Проверяем, есть ли положительные примеры
    if y_t.sum() == 0:
        print(f'  ⚠️ Нет положительных примеров, берем все признаки')
        selected_features_dict[t] = X_train.columns.tolist()[:TOP_K_FEATURES]
        continue
    
    # Для редких классов уменьшаем подвыборку
    pos_ratio = y_t.sum() / len(y_t)
    if pos_ratio < 0.01:
        subsample_size = min(SUBSAMPLE_SIZE, len(y_t) * 20)  # Не более 20x от числа положительных
        print(f'    Редкий класс (ratio={pos_ratio:.4f}), subsample_size={subsample_size}')
    else:
        subsample_size = SUBSAMPLE_SIZE
    
    selected_features_dict[t] = select_features_for_target(
        t, X_train, y_t, cat_feature_names, 
        k=TOP_K_FEATURES, 
        subsample_size=subsample_size
    )

# Статистика по отобранным признакам
print("\nСтатистика отбора признаков:")
selected_counts = []
for t in target_cols:
    n_features = len(selected_features_dict.get(t, []))
    selected_counts.append(n_features)
    if n_features < 100:
        print(f"  ⚠️ {t}: только {n_features} признаков (мало положительных?)")
    elif t in target_cols[:5]:
        print(f"  {t}: {n_features} признаков")

print(f"\nСреднее количество отобранных признаков: {np.mean(selected_counts):.1f}")
print(f"Медиана: {np.median(selected_counts):.1f}")

Конвертация в pandas...
Размер данных: X_train=(750000, 2444), X_test=(250000, 2444)

Отбор признаков для каждого таргета:

[1/41] target_1_1...
  Отбор признаков для target_1_1...
    Подвыборка: 120000 строк (положительных: 7797.0)
    Отобрано 300 признаков

[2/41] target_1_2...
    Редкий класс (ratio=0.0034), subsample_size=120000
  Отбор признаков для target_1_2...
    Подвыборка: 120000 строк (положительных: 2569.0)
    Отобрано 300 признаков

[3/41] target_1_3...
  Отбор признаков для target_1_3...
    Подвыборка: 120000 строк (положительных: 17839.0)
    Отобрано 300 признаков

[4/41] target_1_4...
  Отбор признаков для target_1_4...
    Подвыборка: 120000 строк (положительных: 17572.0)
    Отобрано 300 признаков

[5/41] target_1_5...
    Редкий класс (ratio=0.0018), subsample_size=120000
  Отбор признаков для target_1_5...
    Подвыборка: 120000 строк (положительных: 1379.0)
    Отобрано 300 признаков

[6/41] target_2_1...
    Редкий класс (ratio=0.0071), subsample_size=12000

## 4. Функция обучения одного LightGBM

In [15]:
def train_lgbm(params, model_name='LGB'):
    """Обучает LightGBM с K-Fold, используя отобранные для каждого таргета признаки."""
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    oof  = np.zeros((len(X_train), len(target_cols)))
    pred = np.zeros((len(X_test),  len(target_cols)))
    
    for t_idx, target_name in enumerate(target_cols):
        y = y_train[target_name].values
        selected = selected_features_dict[target_name]
        X_tr_sel = X_train[selected]
        X_te_sel = X_test[selected]
        # категориальные из отобранных
        cat_sel = [c for c in cat_feature_names if c in selected]
        
        fold_preds_test = []
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr_sel)):
            X_tr, X_val = X_tr_sel.iloc[tr_idx], X_tr_sel.iloc[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            
            model = lgb.LGBMClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(-1)],
                categorical_feature=cat_sel,
            )
            oof[val_idx, t_idx] = model.predict_proba(X_val)[:, 1]
            fold_preds_test.append(model.predict_proba(X_te_sel)[:, 1])
            del model
            gc.collect()
        
        pred[:, t_idx] = np.mean(fold_preds_test, axis=0)
        
        auc = roc_auc_score(y, oof[:, t_idx])
        print(f'  [{t_idx+1:02d}/{len(target_cols)}] {target_name:15s}  AUC = {auc:.4f}')
    
    macro_auc = roc_auc_score(y_train.values, oof, average='macro')
    print(f'\n=== {model_name} OOF Macro AUC = {macro_auc:.5f} ===')
    return oof, pred, macro_auc

## 5. LightGBM-1 — широкие деревья, быстрый

In [ ]:
base_params = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'random_state': SEED,
    'verbose': -1,
}
USE_GPU = True
if USE_GPU:
    base_params['device'] = 'gpu'
    base_params['gpu_platform_id'] = 0
    base_params['gpu_device_id'] = 0

PARAMS_1 = base_params.copy()

print("Обучаем LightGBM-1 (только отобранные признаки)...")
oof_1, pred_1, auc_1 = train_lgbm(PARAMS_1, 'LGB-1')

np.save(f'{OUT_DIR}oof_1.npy',  oof_1)
np.save(f'{OUT_DIR}pred_1.npy', pred_1)
print('Сохранено: oof_1.npy, pred_1.npy')

Обучаем LightGBM-1 (только отобранные признаки)...
  [01/41] target_1_1       AUC = 0.9105
  [02/41] target_1_2       AUC = 0.8123
  [03/41] target_1_3       AUC = 0.8669
  [04/41] target_1_4       AUC = 0.8286
  [05/41] target_1_5       AUC = 0.8782
  [06/41] target_2_1       AUC = 0.8180
  [07/41] target_2_2       AUC = 0.9334
  [08/41] target_2_3       AUC = 0.7838
  [09/41] target_2_4       AUC = 0.7347
  [10/41] target_2_5       AUC = 0.7108
  [11/41] target_2_6       AUC = 0.7285
  [12/41] target_2_7       AUC = 0.8166
  [13/41] target_2_8       AUC = 0.8785
  [14/41] target_3_1       AUC = 0.6844
  [15/41] target_3_2       AUC = 0.9114
  [16/41] target_3_3       AUC = 0.7432
  [17/41] target_3_4       AUC = 0.8881
  [18/41] target_3_5       AUC = 0.9630
  [19/41] target_4_1       AUC = 0.8403
  [20/41] target_5_1       AUC = 0.7366
  [21/41] target_5_2       AUC = 0.6863
  [22/41] target_6_1       AUC = 0.7138
  [23/41] target_6_2       AUC = 0.7132
  [24/41] target_6_3       AU

## 6. LightGBM-2 — глубокие деревья, медленный learning rate

In [ ]:
PARAMS_2 = {
    **base_params,
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'min_child_samples': 20,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.05,
    'reg_lambda': 2.0,
    'random_state': SEED + 1,
}

print("Обучаем LightGBM-2 (только отобранные признаки)...")
oof_2, pred_2, auc_2 = train_lgbm(PARAMS_2, 'LGB-2')

np.save(f'{OUT_DIR}oof_2.npy',  oof_2)
np.save(f'{OUT_DIR}pred_2.npy', pred_2)
print('Сохранено: oof_2.npy, pred_2.npy')

Обучаем LightGBM-2 (только отобранные признаки)...
  [01/41] target_1_1       AUC = 0.9160
  [02/41] target_1_2       AUC = 0.8234
  [03/41] target_1_3       AUC = 0.8712
  [04/41] target_1_4       AUC = 0.8332
  [05/41] target_1_5       AUC = 0.8944
  [06/41] target_2_1       AUC = 0.8213
  [07/41] target_2_2       AUC = 0.9364
  [08/41] target_2_3       AUC = 0.7830
  [09/41] target_2_4       AUC = 0.7403
  [10/41] target_2_5       AUC = 0.6748
  [11/41] target_2_6       AUC = 0.7270
  [12/41] target_2_7       AUC = 0.8359
  [13/41] target_2_8       AUC = 0.9617
  [14/41] target_3_1       AUC = 0.6929
  [15/41] target_3_2       AUC = 0.9139
  [16/41] target_3_3       AUC = 0.7542
  [17/41] target_3_4       AUC = 0.9447
  [18/41] target_3_5       AUC = 0.9686
  [19/41] target_4_1       AUC = 0.8401
  [20/41] target_5_1       AUC = 0.7316
  [21/41] target_5_2       AUC = 0.6826
  [22/41] target_6_1       AUC = 0.7130
  [23/41] target_6_2       AUC = 0.7137
  [24/41] target_6_3       AU

In [ ]:
PARAMS_3 = {
    **base_params,
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'num_leaves': 64,
    'min_child_samples': 30,
    'subsample': 0.9,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.2,
    'reg_lambda': 1.5,
    'random_state': SEED + 2
}

print("Обучаем LightGBM-3 (только отобранные признаки)...")
oof_3, pred_3, auc_3 = train_lgbm(PARAMS_3, 'LGB-3')

np.save(f'{OUT_DIR}oof_3.npy',  oof_3)
np.save(f'{OUT_DIR}pred_3.npy', pred_3)
print('Сохранено: oof_3.npy, pred_3.npy')

Обучаем LightGBM-3 (только отобранные признаки)...
  [01/41] target_1_1       AUC = 0.9174
  [02/41] target_1_2       AUC = 0.8268
  [03/41] target_1_3       AUC = 0.8721
  [04/41] target_1_4       AUC = 0.8349
  [05/41] target_1_5       AUC = 0.9011


In [ ]:
PARAMS_4 = {
    **base_params,
    'n_estimators': 800,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 30,
    'subsample': 0.9,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.2,
    'reg_lambda': 1.5,
    'random_state': SEED + 3
}

print("Обучаем LightGBM-3 (только отобранные признаки)...")
oof_4, pred_4, auc_4 = train_lgbm(PARAMS_4, 'LGB-4')

np.save(f'{OUT_DIR}oof_4.npy',  oof_4)
np.save(f'{OUT_DIR}pred_4.npy', pred_4)
print('Сохранено: oof_4.npy, pred_4.npy')

## 7. Ансамбль

In [ ]:
total = auc_1 + auc_2 + auc_3 + auc_4
w1, w2, w3, w4 = auc_1 / total, auc_2 / total, auc_3/total, auc_4/total
oof_blend = w1 * oof_1 + w2 * oof_2 + w3 * oof_3 + w4 * oof_4
pred_blend = w1 * pred_1 + w2 * pred_2 + w3 * pred_3 + w4 * pred_4
auc_blend = roc_auc_score(y_train.values, oof_blend, average='macro')

print(f'LGB-1     OOF Macro AUC = {auc_1:.5f}  (вес = {w1:.3f})')
print(f'LGB-2     OOF Macro AUC = {auc_2:.5f}  (вес = {w2:.3f})')
print(f'LGB-3     OOF Macro AUC = {auc_3:.5f}  (вес = {w3:.3f})')
print(f'LGB-4     OOF Macro AUC = {auc_4:.5f}  (вес = {w4:.3f})')
print(f'Ансамбль  OOF Macro AUC = {auc_blend:.5f}')

LGB-1     OOF Macro AUC = 0.81347  (вес = 0.498)
LGB-2     OOF Macro AUC = 0.82009  (вес = 0.502)
Ансамбль  OOF Macro AUC = 0.82302


## 8. Формирование и проверка сабмита

In [19]:
predict_schema = [c.replace('target_', 'predict_') for c in target_cols]

# Создаем DataFrame с предсказаниями
pred_df = pl.DataFrame(pred_blend, schema=predict_schema)

# Добавляем customer_id
submit = pl.DataFrame({'customer_id': test_customer_ids['customer_id']}).hstack(pred_df)

# Сохраняем
submit.write_parquet(f'{OUT_DIR}submission.parquet')
print(f"✅ Сабмит сохранён: {OUT_DIR}submission.parquet")
print(f"   Размер: {submit.shape[0]} строк, {submit.shape[1]} колонок")
print("\nПервые 3 строки сабмита:")
print(submit.head(3))

# Для отображения в Jupyter
try:
    from IPython.display import FileLink
    display(FileLink(f'{OUT_DIR}submission.parquet'))
except:
    print(f"Файл: {OUT_DIR}submission.parquet")

✅ Сабмит сохранён: data/submission.parquet
   Размер: 250000 строк, 42 колонок

Первые 3 строки сабмита:
shape: (3, 42)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ customer_ ┆ predict_1 ┆ predict_1 ┆ predict_1 ┆ … ┆ predict_9 ┆ predict_9 ┆ predict_9 ┆ predict_ │
│ id        ┆ _1        ┆ _2        ┆ _3        ┆   ┆ _6        ┆ _7        ┆ _8        ┆ 10_1     │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ i32       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1750001   ┆ 0.000737  ┆ 0.001822  ┆ 0.007611  ┆ … ┆ 0.423848  ┆ 0.081794  ┆ 0.00087   ┆ 0.371872 │
│ 1750002   ┆ 0.007398  ┆ 0.003442  ┆ 0.046927  ┆ … ┆ 0.312979  ┆ 0.106565  ┆ 0.000534  ┆ 0.230498 │
│ 1750003   ┆ 0.001222  ┆ 0.002179  ┆ 0.009184  ┆ … ┆ 0.258224  ┆ 0.0406

c:\Users\PC\Desktop\ml\data\submission.parquet